In [4]:
%%writefile lab9_npc.py
"""lab9_npc.py - offline stand-in model and game data for Lab 9 (LangChain NPC).

The class StandInNPC is a real LangChain chat model, so prompts, chains, parsers,
tools and message history all work on it exactly as they would on a hosted model.
It is NOT a language model: it uses keyword rules and seeded random numbers so that
every student gets the same output. Do not edit this file.

To use a real model later, replace the single line  llm = StandInNPC()  with, for
example,  llm = ChatOllama(model="llama3.2:3b")  or  llm = ChatGroq(model=...).
"""
import random
import re

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

LOG = []                      # every model call: prompt text and reply
_STATE = {"rng": random.Random(0), "mode": "normal"}


def set_seed(seed, mode="normal"):
    """Seed the stand-in. mode='stuck' makes it request a tool forever."""
    _STATE["rng"] = random.Random(seed)
    _STATE["mode"] = mode


# ------------------------------------------------------------------ game data
SHOP = {"iron sword": (2, 50), "health potion": (0, 15), "oak shield": (1, 40)}


@tool
def check_inventory(item: str) -> str:
    """Check how many of an item the shop has in stock and its price in gold."""
    stock, price = SHOP.get(item.lower(), (0, 0))
    if item.lower() not in SHOP:
        return f"{item}: not sold here"
    return f"{item.lower()}: {stock} in stock, {price} gold"


@tool
def roll_dice(sides: int = 20) -> str:
    """Roll one die with the given number of sides and return the number."""
    return str(_STATE["rng"].randint(1, int(sides)))


# ------------------------------------------------------- stand-in behaviour
PREFIX = {"grumpy": "*grunts* ", "cheerful": "Ha! ", "nervous": "Um... "}
EMOTION = {"grumpy": "angry", "cheerful": "happy", "nervous": "afraid"}


def _persona(system):
    m = re.search(r"You are (\w+), an? ([^.]+)\. Mood: (\w+)", system)
    return (m.group(1), m.group(2), m.group(3)) if m else ("Stranger", "villager", "calm")


def _find(texts, pattern):
    found = None
    for t in texts:
        m = re.search(pattern, t, re.I)
        if m:
            found = m.group(1)
    return found


def _line(system, humans, text):
    name, role, mood = _persona(system)
    pre, low = PREFIX.get(mood, ""), text.lower()
    if "what is my name" in low:
        who = _find(humans, r"my name is (\w+)")
        return pre + (f"You told me your name is {who}." if who else "You have not told me your name yet.")
    if "what did i" in low:
        what = _find(humans, r"i (?:chose|decided) to ([^.!?]+)")
        return pre + (f"You chose to {what}. I remember." if what else "You have not told me of any choice.")
    if "my name is" in low:
        return pre + f"Well met, {_find([text], r'my name is (\w+)')}. I am {name} the {role}."
    if re.search(r"i (chose|decided) to", low):
        return pre + "A bold choice. I will remember it."
    if "quest" in low or "work" in low:
        return pre + "Wolves are raiding the mill. Bring me three pelts."
    return pre + f"I am {name} the {role}. What do you need?"


def _item_in(text):
    for item in SHOP:
        if item in text or item.split()[-1] in text:
            return item
    return "iron sword"


def _respond(messages, tools):
    rng = _STATE["rng"]
    system = next((m.content for m in messages if m.type == "system"), "")
    humans = [m.content for m in messages if m.type == "human"]
    last = messages[-1]
    pre = PREFIX.get(_persona(system)[2], "")
    n_calls = sum(1 for m in messages if m.type == "ai" and getattr(m, "tool_calls", None))

    if tools and _STATE["mode"] == "stuck":
        return {"name": "roll_dice", "args": {"sides": 20}, "id": f"call_{n_calls}"}
    if last.type == "tool":
        if last.name == "check_inventory":
            if "0 in stock" in last.content:
                return pre + "Sold out. Come back after the next delivery."
            return pre + f"Yes. {last.content}."
        if last.name == "roll_dice":
            n = int(last.content)
            return pre + f"You rolled {n}. " + ("A solid hit!" if n >= 11 else "You miss.")
        return pre + "I cannot do that."
    text = humans[-1] if humans else ""
    low = text.lower()
    if tools:
        if re.search(r"\b(buy|price|cost|stock|sword|shield|potion)\b", low):
            if rng.random() < 0.15:
                return {"name": "forge_item", "args": {"item": _item_in(low)}, "id": f"call_{n_calls}"}
            return {"name": "check_inventory", "args": {"item": _item_in(low)}, "id": f"call_{n_calls}"}
        if re.search(r"\b(roll|attack|dice)\b", low):
            return {"name": "roll_dice", "args": {"sides": 20}, "id": f"call_{n_calls}"}
    if not system:
        return "Hello from the Lab 9 stand-in. I am a Python function, not a language model."
    line = _line(system, humans, text)
    if "JSON" in system:
        mood = _persona(system)[2]
        reply = ('{"say": "%s", "emotion": "%s", "action": "%s"}'
                 % (line.replace('"', "'"), EMOTION.get(mood, "neutral"),
                    "offer_quest" if "quest" in low else "none"))
        r = rng.random()
        if r < 0.25:
            return "Here is the reply: " + reply + " Hope this helps!"
        if r < 0.40:
            return reply.replace(', "action": "none"', "").replace(', "action": "offer_quest"', "")
        return reply
    return line


class StandInNPC(BaseChatModel):
    """Offline stand-in chat model. Keyword rules plus seeded randomness."""

    @property
    def _llm_type(self):
        return "stand-in-npc"

    def bind_tools(self, tools, **kwargs):
        return self.bind(tools=[convert_to_openai_tool(t) for t in tools], **kwargs)

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        reply = _respond(messages, kwargs.get("tools"))
        LOG.append({"prompt": "\n".join(f"{m.type}: {m.content}" for m in messages), "reply": reply})
        if isinstance(reply, str):
            msg = AIMessage(content=reply)
        else:
            msg = AIMessage(content="", tool_calls=[{**reply, "type": "tool_call"}])
        return ChatResult(generations=[ChatGeneration(message=msg)])


llm = StandInNPC()


Overwriting lab9_npc.py


# Lab 9: LangChain NPC - Hand-Rolled vs Framework
CGD-460L Agentic AI | Fall 2026 | Week 9. Offline stand-in model, no API key needed.

In [5]:
# Setup
import json
import inspect

import langchain
import langchain_core
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.exceptions import OutputParserException
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda
from pydantic import BaseModel
from typing import Literal

import lab9_npc as npc
from lab9_npc import llm, set_seed, check_inventory, roll_dice

print("langchain-core", langchain_core.__version__, "| langchain", langchain.__version__)


langchain-core 1.7.0 | langchain 1.4.7


In [6]:
# Activity 1 - Call the model and log the call
npc.LOG.clear()
set_seed(0)
reply = llm.invoke("Say hello in one short sentence.")
print(type(reply).__name__, "|", reply.content)
print("calls logged:", len(npc.LOG))


AIMessage | Hello from the Lab 9 stand-in. I am a Python function, not a language model.
calls logged: 1


In [7]:
# Activity 2 - The hand-rolled NPC
SYSTEM = "You are {npc}, a {role}. Mood: {mood}. Stay in character and answer in one or two sentences."

def hand_npc(player_text, name="Brom", role="gruff blacksmith", mood="grumpy"):
    system = SYSTEM.format(npc=name, role=role, mood=mood)
    messages = [SystemMessage(content=system), HumanMessage(content=player_text)]
    return llm.invoke(messages).content.strip()

for line in ["Hello there.", "Do you have any work for me?"]:
    print("Player:", line)
    print("  Brom:", hand_npc(line))


Player: Hello there.
  Brom: *grunts* I am Brom the gruff blacksmith. What do you need?
Player: Do you have any work for me?
  Brom: *grunts* Wolves are raiding the mill. Bring me three pelts.


In [8]:
# Activity 3 - The same NPC as a LangChain chain
prompt = ChatPromptTemplate.from_messages([("system", SYSTEM), ("human", "{player}")])
npc_chain = prompt | llm | StrOutputParser()

BROM = {"npc": "Brom", "role": "gruff blacksmith", "mood": "grumpy"}
for line in ["Hello there.", "Do you have any work for me?"]:
    print("Player:", line)
    print("  Brom:", npc_chain.invoke({**BROM, "player": line}))
print("same answers as the hand-rolled version:",
      npc_chain.invoke({**BROM, "player": "Hello there."}) == hand_npc("Hello there."))


Player: Hello there.
  Brom: *grunts* I am Brom the gruff blacksmith. What do you need?
Player: Do you have any work for me?
  Brom: *grunts* Wolves are raiding the mill. Bring me three pelts.
same answers as the hand-rolled version: True


In [9]:
# Activity 4 - Batch: three player lines in one call
lines = ["Hello there.", "Do you have any work for me?", "My name is Aria."]
npc.LOG.clear()
answers = npc_chain.batch([{**BROM, "player": x} for x in lines])
for x, a in zip(lines, answers):
    print(f"{x:30} -> {a}")
print("model calls:", len(npc.LOG))


Hello there.                   -> *grunts* I am Brom the gruff blacksmith. What do you need?
Do you have any work for me?   -> *grunts* Wolves are raiding the mill. Bring me three pelts.
My name is Aria.               -> *grunts* Well met, Aria. I am Brom the gruff blacksmith.
model calls: 3


In [10]:
PERSONAS = [
    {"npc": "Brom", "role": "gruff blacksmith", "mood": "grumpy"},
    {"npc": "Lyra", "role": "village healer", "mood": "cheerful"},
    {"npc": "Pip", "role": "shy messenger", "mood": "nervous"},

    # TODO 2: Fourth persona
    {"npc": "Sara", "role": "village guard", "mood": "cheerful"}
]

# TODO 1: Loop through all personas
for persona in PERSONAS:
    reply = npc_chain.invoke({
        **persona,
        "player": "Do you have any work for me?"
    })

    print(f"{persona['npc']} ({persona['mood']}): {reply}")

Brom (grumpy): *grunts* Wolves are raiding the mill. Bring me three pelts.
Lyra (cheerful): Ha! Wolves are raiding the mill. Bring me three pelts.
Pip (nervous): Um... Wolves are raiding the mill. Bring me three pelts.
Sara (cheerful): Ha! Wolves are raiding the mill. Bring me three pelts.


In [11]:

# Task 2 - Structured replies, parsed and retried

class NPCReply(BaseModel):
    say: str
    emotion: Literal["angry", "happy", "afraid", "neutral"]
    action: Literal["none", "offer_quest", "open_shop"]


parser = PydanticOutputParser(pydantic_object=NPCReply)

json_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM + "\n{format_instructions}"),
    ("human", "{player}")
]).partial(
    format_instructions=parser.get_format_instructions()
)

# TODO 1: Create JSON chain
json_chain = json_prompt | llm | parser

# TODO 2: Add retry
safe_chain = json_chain.with_retry(
    retry_if_exception_type=(OutputParserException,),
    stop_after_attempt=3,
    wait_exponential_jitter=False
)


# TODO 3: Measure usable replies and failures
def measure(chain, seeds=range(20)):
    ok = 0
    failed = 0

    npc.LOG.clear()

    for seed in seeds:
        set_seed(seed)

        for x in lines:
            try:
                chain.invoke({
                    **BROM,
                    "player": x
                })
                ok += 1

            except OutputParserException:
                failed += 1

    return ok, failed, len(npc.LOG)


# TODO 4: Test first four seeds using safe_chain
for seed in range(4):
    set_seed(seed)

    try:
        reply = safe_chain.invoke({
            **BROM,
            "player": lines[0]
        })

        print(f"seed {seed}: accepted -> {reply}")

    except OutputParserException as err:
        print(
            f"seed {seed}: REJECTED -> "
            f"{str(err).splitlines()[0][:60]}"
        )


# Compare without retry and with retry
for name, chain in [
    ("no retry", json_chain),
    ("with retry", safe_chain)
]:
    ok, failed, calls = measure(chain)

    print(
        f"{name:10}: usable {ok}/60 "
        f"({ok / 60:.0%}), failed {failed}, "
        f"model calls {calls}"
    )



seed 0: accepted -> say='*grunts* I am Brom the gruff blacksmith. What do you need?' emotion='angry' action='none'
seed 1: accepted -> say='*grunts* I am Brom the gruff blacksmith. What do you need?' emotion='angry' action='none'
seed 2: accepted -> say='*grunts* I am Brom the gruff blacksmith. What do you need?' emotion='angry' action='none'
seed 3: accepted -> say='*grunts* I am Brom the gruff blacksmith. What do you need?' emotion='angry' action='none'
no retry  : usable 41/60 (68%), failed 19, model calls 60
with retry: usable 59/60 (98%), failed 1, model calls 90


In [12]:

# Task 3 - Tools and a safe agent loop

TOOLS = {
    "check_inventory": check_inventory,
    "roll_dice": roll_dice
}


def tool_npc(player_text, max_steps=4):

    messages = [
        SystemMessage(content=SYSTEM.format(**BROM)),
        HumanMessage(content=player_text)
    ]

    bound = llm.bind_tools(list(TOOLS.values()))
    trace = []

    for step in range(1, max_steps + 1):

        ai = bound.invoke(messages)
        messages.append(ai)

        # TODO 1: Stop if model gives normal text
        if not ai.tool_calls:
            return ai.content, trace + [f"step {step}: final answer"]

        # TODO 2: Check and execute tools
        for call in ai.tool_calls:

            name = call["name"]

            if name not in TOOLS:
                result = f"unknown tool {name}. Allowed: {', '.join(TOOLS.keys())}"

            else:
                result = TOOLS[name].invoke(call["args"])

            # TODO 3: Save result in trace and messages
            trace.append(
                f"step {step}: {name}({call['args']}) -> {result}"
            )

            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=call["id"],
                    name=name
                )
            )

    return None, trace + ["stopped: step limit reached"]


# Testing
set_seed(2)

for q in [
    "How much is the iron sword?",
    "I attack the goblin!",
    "Can I buy a health potion?"
]:

    answer, trace = tool_npc(q)

    print("Player:", q)
    print("\n".join("  " + t for t in trace))
    print("  Brom:", answer)


# Test 40 runs
counts = {
    "answered": 0,
    "refused unknown tool": 0,
    "step limit": 0
}

for seed in range(40):

    set_seed(seed)

    answer, trace = tool_npc("How much is the iron sword?")

    counts["answered" if answer else "step limit"] += 1

    counts["refused unknown tool"] += any(
        "unknown tool" in t for t in trace
    )

print("40 runs:", counts)


# Test stuck model
set_seed(0, mode="stuck")

answer, trace = tool_npc("Hello")

print(
    "stuck model:",
    answer,
    "|",
    trace[-1],
    "| tool calls made:",
    len(trace) - 1
)



Player: How much is the iron sword?
  step 1: check_inventory({'item': 'iron sword'}) -> iron sword: 2 in stock, 50 gold
  step 2: final answer
  Brom: *grunts* Yes. iron sword: 2 in stock, 50 gold.
Player: I attack the goblin!
  step 1: roll_dice({'sides': 20}) -> 2
  step 2: final answer
  Brom: *grunts* You rolled 2. You miss.
Player: Can I buy a health potion?
  step 1: forge_item({'item': 'health potion'}) -> unknown tool forge_item. Allowed: check_inventory, roll_dice
  step 2: final answer
  Brom: *grunts* I cannot do that.
40 runs: {'answered': 40, 'refused unknown tool': 5, 'step limit': 0}
stuck model: None | stopped: step limit reached | tool calls made: 4


In [14]:

# Task 4 - Memory across turns

from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

mem_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{player}")
])

mem_chain = mem_prompt | llm | StrOutputParser()

HISTORY = {}  # One history list per game session


def say(session, text):

    # TODO 1: Get or create history
    history = HISTORY.setdefault(session, [])

    # TODO 2: Call the memory chain
    reply = mem_chain.invoke({
        **BROM,
        "player": text,
        "history": history
    })

    # TODO 3: Save player and NPC messages
    history.append(HumanMessage(content=text))
    history.append(AIMessage(content=reply))

    # TODO 4: Return reply
    return reply


# Testing
for s, text in [
    ("game1", "My name is Aria."),
    ("game1", "I decided to spare the bandit."),
    ("game1", "What is my name?"),
    ("game1", "What did I decide?"),
    ("game2", "What is my name?")
]:

    print(f"[{s}] Player: {text}\n        Brom: {say(s, text)}")


print(
    "messages stored in game1:",
    len(HISTORY["game1"]),
    "| in game2:",
    len(HISTORY["game2"])
)


[game1] Player: My name is Aria.
        Brom: *grunts* Well met, Aria. I am Brom the gruff blacksmith.
[game1] Player: I decided to spare the bandit.
        Brom: *grunts* A bold choice. I will remember it.
[game1] Player: What is my name?
        Brom: *grunts* You told me your name is Aria.
[game1] Player: What did I decide?
        Brom: *grunts* You chose to spare the bandit. I remember.
[game2] Player: What is my name?
        Brom: *grunts* You have not told me your name yet.
messages stored in game1: 8 | in game2: 2


In [17]:

# Task 5 - The companion: routing, tools and memory together

# TODO 1: Create router
def needs_tool(text):
    low = text.lower()
    return any(w in low for w in [
        "buy", "price", "cost", "sword",
        "potion", "shield", "attack", "roll"
    ])


def route(inputs):
    if needs_tool(inputs["player"]):
        return RunnableLambda(
            lambda i: tool_npc(i["player"])[0]
        )

    return RunnableLambda(
        lambda i: say(i["session"], i["player"])
    )


companion = RunnableLambda(route)


# Test conversation
SCRIPT = [
    ("My name is Aria.", "Aria"),
    ("I chose to guard the village.", "remember"),
    ("How much is the iron sword?", "gold"),
    ("I attack the wolf!", "rolled"),
    ("What is my name?", "Aria"),
    ("What did I decide?", "guard the village")
]


# TODO 2: Run every line through companion
HISTORY.clear()
npc.LOG.clear()
set_seed(5)

passed = 0

for text, must_contain in SCRIPT:

    out = companion.invoke({
        "player": text,
        "session": "run1"
    }) or ""

    ok = must_contain.lower() in out.lower()
    passed += ok

    print(f"{'PASS' if ok else 'FAIL'} | {text:30} -> {out}")


print(
    f"checks passed: {passed}/6 | model calls: {len(npc.LOG)}"
)



PASS | My name is Aria.               -> *grunts* Well met, Aria. I am Brom the gruff blacksmith.
PASS | I chose to guard the village.  -> *grunts* A bold choice. I will remember it.
PASS | How much is the iron sword?    -> *grunts* Yes. iron sword: 2 in stock, 50 gold.
PASS | I attack the wolf!             -> *grunts* You rolled 12. A solid hit!
PASS | What is my name?               -> *grunts* You told me your name is Aria.
PASS | What did I decide?             -> *grunts* You chose to guard the village. I remember.
checks passed: 6/6 | model calls: 8


In [ ]:
# Task 5b - Measure: hand-rolled versus framework
def hand_json(player_text, tries=3):
    system = SYSTEM.format(**BROM) + "\nReply with JSON: " +'{"say": "...", "emotion": "...", "action": "..."}'
    for _ in range(tries):
        raw = llm.invoke([SystemMessage(content=system), HumanMessage(content=player_text)]).content
        try:
            return NPCReply(**json.loads(raw))
        except Exception:
            continue
    return None

npc.LOG.clear()
ok = 0
for seed in range(20):
    set_seed(seed)
    for x in lines:
        ok += hand_json(x) is not None
print(f"hand-rolled : usable {ok}/60, model calls {len(npc.LOG)}, source lines {len(inspect.getsource(hand_json).splitlines())}")
ok, failed, calls = measure(safe_chain)
print(f"framework   : usable {ok}/60, model calls {calls}")
print("framework also gives: batch ->", hasattr(safe_chain, "batch"),
      "| stream ->", hasattr(safe_chain, "stream"), "| swap the model by changing one line")
